In [ ]:
import time
from pathlib import Path

import polars as pl

In [ ]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data" / "sample_train_data"
PROCESSED_DIR = ROOT / "data" / "processed"
RAW_APPS = RAW_DIR / "sample_train_data.csv"
RAW_LABELS = RAW_DIR / "sample_train_labels.csv"

pl.__version__

'2.0.0'

In [ ]:
inferred = pl.scan_csv(RAW_APPS, separator=";").collect_schema()
print(len(inferred), "columns")
print({str(dtype) for dtype in inferred.values()})
dict(list(inferred.items())[:4])

161 columns
{'Int64', 'String'}


{'IDX': Int64, 'NUM': Int64, 'COL0': String, 'COL1': Int64}

In [ ]:
FEATURE_COLS = [f"COL{i}" for i in range(1, 159)]
APPS_SCHEMA = {
    "IDX": pl.Int64,
    "NUM": pl.UInt8,
    "COL0": pl.String,
    **dict.fromkeys(FEATURE_COLS, pl.UInt8),
}
assert list(APPS_SCHEMA) == list(inferred), "column order mismatch"

apps_lf = pl.scan_csv(RAW_APPS, separator=";", schema=APPS_SCHEMA).select(
    "IDX",
    "NUM",
    pl.col("COL0").str.to_datetime("%Y.%m.%d %H:%M:%S").alias("ts"),
    *FEATURE_COLS,
)
labels_lf = pl.scan_csv(RAW_LABELS, separator=";", schema={"IDX": pl.Int64, "TRG": pl.UInt8})

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
apps_pq = PROCESSED_DIR / "applications.parquet"
labels_pq = PROCESSED_DIR / "labels.parquet"

t0 = time.perf_counter()
apps_lf.sink_parquet(apps_pq)
labels_lf.sink_parquet(labels_pq)
print(
    f"{time.perf_counter() - t0:.1f}s, applications.parquet = {apps_pq.stat().st_size / 2**20:.0f} MB"
)

5.3s, applications.parquet = 91 MB


In [ ]:
t0 = time.perf_counter()
apps = pl.read_parquet(apps_pq)
labels = pl.read_parquet(labels_pq)
print(f"{time.perf_counter() - t0:.1f}s, apps in RAM = {apps.estimated_size('mb'):.0f} MB")
apps.shape, labels.shape

0.3s, apps in RAM = 850 MB


((5090423, 161), (794201, 2))

In [ ]:
assert apps.height == 5_090_423 and labels.height == 794_201

value_check = apps.select(
    nulls=pl.sum_horizontal(pl.all().null_count()),
    max_feature=pl.max_horizontal(pl.col(FEATURE_COLS).max()),
    ts_min=pl.col("ts").min(),
    ts_max=pl.col("ts").max(),
)

per_client = apps.group_by("IDX").agg(
    n=pl.len(),
    num_ok=(pl.col("NUM").sort() == pl.int_range(pl.len())).all(),
    ts_desc=(pl.col("ts").sort_by("NUM").diff().drop_nulls() <= pl.duration(seconds=0)).all(),
    ts_ties=(pl.col("ts").sort_by("NUM").diff().drop_nulls() == pl.duration(seconds=0)).sum(),
)

integrity = per_client.select(
    n_clients=pl.len(),
    bad_length=(~pl.col("n").is_between(2, 10)).sum(),
    bad_num=(~pl.col("num_ok")).sum(),
    ts_not_desc=(~pl.col("ts_desc")).sum(),
    clients_with_ts_ties=(pl.col("ts_ties") > 0).sum(),
    unlabled=pl.col("IDX").is_in(labels["IDX"].implode()).not_().sum(),
)

label_check = labels.select(
    duplicated_idx=pl.col("IDX").is_duplicated().sum(),
    without_data=(~pl.col("IDX").is_in(per_client["IDX"].implode())).sum(),
    target_values=pl.col("TRG").unique().sort().implode(),
)

value_check, integrity, label_check

(shape: (1, 4)
 ┌───────┬─────────────┬─────────────────────┬─────────────────────┐
 │ nulls ┆ max_feature ┆ ts_min              ┆ ts_max              │
 │ ---   ┆ ---         ┆ ---                 ┆ ---                 │
 │ u32   ┆ u8          ┆ datetime[μs]        ┆ datetime[μs]        │
 ╞═══════╪═════════════╪═════════════════════╪═════════════════════╡
 │ 0     ┆ 1           ┆ 2001-12-17 12:30:47 ┆ 2018-12-31 23:52:36 │
 └───────┴─────────────┴─────────────────────┴─────────────────────┘,
 shape: (1, 6)
 ┌───────────┬────────────┬─────────┬─────────────┬──────────────────────┬──────────┐
 │ n_clients ┆ bad_length ┆ bad_num ┆ ts_not_desc ┆ clients_with_ts_ties ┆ unlabled │
 │ ---       ┆ ---        ┆ ---     ┆ ---         ┆ ---                  ┆ ---      │
 │ u32       ┆ u32        ┆ u32     ┆ u32         ┆ u32                  ┆ u32      │
 ╞═══════════╪════════════╪═════════╪═════════════╪══════════════════════╪══════════╡
 │ 794201    ┆ 0          ┆ 0       ┆ 0           ┆ 0   